# 무신사

In [ ]:
import requests
import pandas as pd
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    "Referer": "https://www.musinsa.com/",
    "Origin": "https://www.musinsa.com",
}

BRAND        = "filluminate"
CATEGORY     = "003"
CATEGORY_NAME = "바지"
MAX_GOODS    = 10   # 상위 10개 상품만
MAX_REVIEWS  = 30   # 상품당 최대 30개 리뷰

# ==========================================
# Step 1: 브랜드 상품 수집 (리뷰 많은 순 상위 10개)
# ==========================================

all_data = []
page = 1

while len(all_data) < MAX_GOODS:
    url = "https://api.musinsa.com/api2/dp/v2/plp/goods"
    params = {
        "gf":       "A",
        "sortCode": "POPULAR",
        "category": CATEGORY,
        "brand":    BRAND,
        "size":     30,
        "caller":   "FLAGSHIP",
        "page":     page,
    }
    response = requests.get(url, headers=headers, params=params)

    if response.status_code != 200:
        print(f"상품 수집 실패: {response.status_code}")
        break

    data = response.json().get("data", {})
    goods_list = data.get("list", [])

    if not goods_list:
        break

    for item in goods_list:
        all_data.append({
            "플랫폼":    "무신사",
            "카테고리":  CATEGORY_NAME,
            "브랜드":    item.get("brandName", ""),
            "goodsNo":  str(item.get("goodsNo", "")),
            "상품명":    item.get("goodsName", ""),
            "정가":      item.get("normalPrice", 0),
            "판매가":    item.get("price", 0),
            "할인율(%)": item.get("saleRate", 0),
            "리뷰수":    item.get("reviewCount", 0),
            "리뷰점수":  item.get("reviewScore", 0),
        })

    total_pages = data.get("page", {}).get("totalPages", 1)
    if page >= total_pages:
        break

    page += 1
    time.sleep(0.5)

# 리뷰 많은 순으로 정렬 후 상위 10개만
df = pd.DataFrame(all_data)
df = df.sort_values("리뷰수", ascending=False).head(MAX_GOODS).reset_index(drop=True)
df["조회수"]    = 0
df["누적판매수"] = 0

print(f"\n상품 총 {len(df)}개 수집 완료 (리뷰 많은 순)")
print(df[["카테고리", "브랜드", "상품명", "리뷰수"]].to_string(index=False))


# ==========================================
# Step 2: 만족도 파싱 함수
# ==========================================

def parse_satisfaction(survey):
    if not survey:
        return ""
    questions = survey.get("questions", [])
    parts = []
    for q in questions:
        attribute = q.get("attribute", "")
        answers = q.get("answers", [])
        answer_text = ", ".join([a.get("answerShortText", "") for a in answers])
        parts.append(f"{attribute}: {answer_text}")
    return " / ".join(parts)


# ==========================================
# Step 3: 리뷰 수집 함수 (최대 30개)
# ==========================================

def get_reviews(goods_no, target_count, max_pages=3):
    reviews = []

    for page in range(max_pages):
        if len(reviews) >= target_count:
            break

        url = "https://goods.musinsa.com/api2/review/v1/view/list"
        params = {
            "page":              page,
            "pageSize":          10,
            "goodsNo":           goods_no,
            "sort":              "up_cnt_desc",
            "selectedSimilarNo": goods_no,
            "myFilter":          "false",
            "hasPhoto":          "false",
            "isExperience":      "false",
        }

        try:
            r = requests.get(url, headers=headers, params=params, timeout=10)

            if r.status_code != 200:
                print(f"  상태코드 {r.status_code} -> 중단")
                break

            data        = r.json().get("data", {})
            review_list = data.get("list", [])

            if not review_list:
                break

            for review in review_list:
                profile = review.get("userProfileInfo") or {}
                images  = review.get("images") or []

                reviews.append({
                    "goodsNo":  str(goods_no),
                    "리뷰번호": review.get("no", ""),
                    "작성자": profile.get("userNickName", ""),
                    "리뷰내용": review.get("content", ""),
                    "평점":     int(review.get("grade", 0)),
                    "체험단":   review.get("type") == "experience",
                    "구매옵션": review.get("goodsOption", ""),
                    "키":       profile.get("userHeight", ""),
                    "몸무게":   profile.get("userWeight", ""),
                    "성별":     profile.get("reviewSex", ""),
                    "작성일":   review.get("createDate", ""),
                    "만족도":   parse_satisfaction(review.get("reviewSurveySatisfaction")),
                    "사진유무": len(images) > 0,
                    "도움돼요": review.get("likeCount", 0),
                })

                if len(reviews) >= target_count:
                    break

            total_pages = data.get("page", {}).get("totalPages", 0)
            if page >= total_pages - 1:
                break

            time.sleep(0.3)

        except Exception as e:
            print(f"  오류: {e}")
            break

    return reviews


# ==========================================
# Step 4: 상품 통계(조회수, 판매량) 수집 함수
# ==========================================

def get_goods_stats(goods_no):
    url = f"https://goods-detail.musinsa.com/api2/goods/{goods_no}/stat"

    try:
        time.sleep(0.2)
        r = requests.get(url, headers=headers, timeout=5)
        if r.status_code == 200:
            data = r.json()
            sales_count = data.get("data", {}).get("purchaseTotal", 0)
            views_count = data.get("data", {}).get("pageViewTotal", 0)
            return sales_count, views_count
    except Exception as e:
        print(f"통계 수집 오류 ({goods_no}): {e}")

    return 0, 0


# ==========================================
# Step 5: survey 수집 함수
# ==========================================

def get_survey(goods_no):
    url = f"https://goods.musinsa.com/api2/review/v1/view/survey/{goods_no}/summary"

    try:
        time.sleep(0.2)
        r = requests.get(url, headers=headers, timeout=5)
        if r.status_code == 200:
            questions = r.json().get("data", {}).get("questions", [])
            rows = []
            for q in questions:
                attribute = q.get("attribute", "")
                for answer in q.get("answers", []):
                    rows.append({
                        "goodsNo":    str(goods_no),
                        "항목":       attribute,
                        "답변":       answer.get("answerShortText", ""),
                        "비율(%)":    answer.get("percentage", 0),
                        "응답수":     answer.get("count", 0),
                    })
            return rows
    except Exception as e:
        print(f"survey 수집 오류 ({goods_no}): {e}")

    return []


# ==========================================
# Step 6: 전체 상품 리뷰 및 통계 수집
# ==========================================

all_reviews = []
all_surveys = []
total_items = len(df)

for idx, row in df.iterrows():
    goods_no     = row["goodsNo"]
    target_count = min(int(row["리뷰수"]), MAX_REVIEWS)
    current_item = idx + 1

    # 통계 수집
    sales, views = get_goods_stats(goods_no)
    df.at[idx, "누적판매수"] = sales
    df.at[idx, "조회수"]    = views

    # survey 수집
    survey_rows = get_survey(goods_no)
    all_surveys.extend(survey_rows)

    if not goods_no or target_count == 0:
        print(f"[{current_item}/{total_items}] [스킵] {str(row['상품명'])[:25]} (리뷰 없음 | 누적판매: {sales})")
        time.sleep(0.3)
        continue

    print(f"[{current_item}/{total_items}] 리뷰 수집: {str(row['상품명'])[:25]} (목표: {target_count}개 | 누적판매: {sales})")

    reviews = get_reviews(goods_no, target_count=target_count)
    print(f"  -> {len(reviews)}개 수집 / 목표 {target_count}개")
    all_reviews.extend(reviews)

    time.sleep(0.5)

df_reviews = pd.DataFrame(all_reviews) if all_reviews else pd.DataFrame()
df_survey  = pd.DataFrame(all_surveys) if all_surveys else pd.DataFrame()

print(f"\n리뷰 총 {len(df_reviews)}개 수집 완료!")
print(f"survey 총 {len(df_survey)}행 수집 완료!")


# ==========================================
# Step 7: CSV 저장
# ==========================================

# 상품 정보 CSV
df_products = df[[
    "플랫폼", "카테고리", "브랜드", "goodsNo", "상품명",
    "정가", "판매가", "할인율(%)", "조회수", "누적판매수", "리뷰수", "리뷰점수"
]]
df_products.to_csv("musinsa_products.csv", index=False, encoding="utf-8-sig")
print("musinsa_products.csv 저장 완료!")

# 리뷰 CSV
if not df_reviews.empty:
    df_reviews.to_csv("musinsa_reviews.csv", index=False, encoding="utf-8-sig")
    print("musinsa_reviews.csv 저장 완료!")
else:
    print("수집된 리뷰 없음 - musinsa_reviews.csv 저장 안 함")

# survey CSV
if not df_survey.empty:
    df_survey.to_csv("musinsa_survey.csv", index=False, encoding="utf-8-sig")
    print("musinsa_survey.csv 저장 완료!")
else:
    print("수집된 survey 없음 - musinsa_survey.csv 저장 안 함")


상품 총 10개 수집 완료 (리뷰 많은 순)
카테고리    브랜드                                    상품명   리뷰수
  바지 필루미네이트     [해피가이정호 pick] 데미지 워시드 데님 팬츠-미디엄 블루 16190
  바지 필루미네이트                      나일론 투 턱 테이핑 팬츠-블랙  4448
  바지 필루미네이트         [해피가이정호 pick] 데미지 워시드 데님 팬츠-블랙  2774
  바지 필루미네이트 [해피가이정호 pick] 올 데이 린넨 라이크 밴딩 팬츠-5Color  1393
  바지 필루미네이트                       데미지 워시드 데님 팬츠-블루  1293
  바지 필루미네이트   [패션플래닛×필루미네이트] 에센셜 와이드 스웨트 팬츠-3Color   520
  바지 필루미네이트                          밀리터리 카고 팬츠-블랙   400
  바지 필루미네이트     [해피가이정호 pick] 데미지 워시드 데님 팬츠-3Color   139
  바지 필루미네이트                   올 데이 코튼 밴딩 팬츠-3Color   132
  바지 필루미네이트      [해피가이정호 pick] 데미지 워시드 데님 팬츠-로우 블랙    46
[1/10] 리뷰 수집: [해피가이정호 pick] 데미지 워시드 데님  (목표: 30개 | 누적판매: 149468)
  -> 30개 수집 / 목표 30개
[2/10] 리뷰 수집: 나일론 투 턱 테이핑 팬츠-블랙 (목표: 30개 | 누적판매: 33580)
  -> 30개 수집 / 목표 30개
[3/10] 리뷰 수집: [해피가이정호 pick] 데미지 워시드 데님  (목표: 30개 | 누적판매: 25096)
  -> 30개 수집 / 목표 30개
[4/10] 리뷰 수집: [해피가이정호 pick] 올 데이 린넨 라이크 (목표: 30개 | 누적판매: 15064)
  -> 30개 수집 / 목표 30개
[5/10] 리뷰 수집: 

In [9]:
import pandas as pd

df_products = pd.read_csv("musinsa_products.csv")
df_reviews  = pd.read_csv("musinsa_reviews.csv")
df_survey   = pd.read_csv("musinsa_survey.csv")

print("===== 상품 =====")
print(df_products.head())

print("\n===== 리뷰 =====")
print(df_reviews.head())

print("\n===== survey =====")
print(df_survey.head())

===== 상품 =====
   플랫폼 카테고리     브랜드  goodsNo                                     상품명     정가  \
0  무신사   바지  필루미네이트  3791988      [해피가이정호 pick] 데미지 워시드 데님 팬츠-미디엄 블루  53000   
1  무신사   바지  필루미네이트  3792018                       나일론 투 턱 테이핑 팬츠-블랙  58000   
2  무신사   바지  필루미네이트  3791990          [해피가이정호 pick] 데미지 워시드 데님 팬츠-블랙  53000   
3  무신사   바지  필루미네이트  4969807  [해피가이정호 pick] 올 데이 린넨 라이크 밴딩 팬츠-5Color  49000   
4  무신사   바지  필루미네이트  4095002                        데미지 워시드 데님 팬츠-블루  53000   

     판매가  할인율(%)    조회수   누적판매수    리뷰수  리뷰점수  
0  32900      38  73028  149468  16190    96  
1  39000      33   3810   33580   4448    96  
2  32900      38  10365   25096   2774    96  
3  32900      33   6295   15064   1393    94  
4  32900      38   8995   10538   1293    94  

===== 리뷰 =====
   goodsNo       작성자                                               리뷰내용  평점  \
0  3791988  정제된퀸스울니트  스크래치 같은 디테일보다 색감 하나만 보고 기대하며 주문했지만, 실제 색상은 화면처...   2   
1  3791988  heongxcg  웬만하면 좋게 쓰겠는데요.바지가 사진과 보는거랑 너무 다르